# Hand Gesture Controlled Music Synthesizer — Evaluation & Metrics

This notebook contains all the visualizations and quantitative metrics needed for a research paper on the **HandSynth** system. It covers:

1. **Dataset Analysis** — class distribution, sample images, per-hand breakdown
2. **Model Architecture** — summary and parameter count
3. **Training Curves** — loss and accuracy over epochs
4. **Classification Metrics** — confusion matrix, precision/recall/F1, per-class accuracy
5. **ROC & AUC Curves** — one-vs-rest for each gesture class
6. **Inference Latency** — Keras vs TFLite vs rule-based benchmarks
7. **Waveform Visualizations** — synthesizer output for each waveform type
8. **Model Size Comparison** — Keras vs TFLite file sizes

In [ ]:
import os
import sys
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import Counter

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_recall_fscore_support, roc_curve, auc,
    ConfusionMatrixDisplay,
)
from tensorflow.keras.utils import to_categorical

# Project imports
sys.path.insert(0, os.path.abspath("."))
from training.model import build_model, IMG_SIZE, NUM_CLASSES
from src.synth import generate_sine, generate_square, generate_sawtooth
from src.config import SAMPLE_RATE, NOTE_FREQS

# Paths
DATASET_DIR = os.path.join(".", "dataset")
MODEL_PATH  = os.path.join(".", "models", "gesture_model.keras")
TFLITE_PATH = os.path.join(".", "models", "gesture_model.tflite")

CLASS_NAMES = ["0 (Fist)", "1 Finger", "2 Fingers", "3 Fingers", "4 Fingers", "5 (Palm)"]

# Plot style — dark theme to match the app aesthetic
plt.rcParams.update({
    "figure.facecolor": "#0a0a0a",
    "axes.facecolor": "#0a0a0a",
    "axes.edgecolor": "#333",
    "axes.labelcolor": "#ccc",
    "text.color": "#ccc",
    "xtick.color": "#999",
    "ytick.color": "#999",
    "grid.color": "#1a1a1a",
    "figure.dpi": 120,
    "font.family": "monospace",
})
NEON = "#00FF41"
NEON_DIM = "#00AA2A"

print("Setup complete.")

---
## 1. Dataset Analysis

In [ ]:
def load_dataset_with_meta():
    """Load dataset and track per-hand counts."""
    images, labels = [], []
    hand_counts = {"left": Counter(), "right": Counter()}
    
    for hand in ("left", "right"):
        hand_dir = os.path.join(DATASET_DIR, hand)
        if not os.path.exists(hand_dir):
            continue
        for class_id in range(NUM_CLASSES):
            class_dir = os.path.join(hand_dir, str(class_id))
            if not os.path.exists(class_dir):
                continue
            files = [f for f in os.listdir(class_dir) if f.endswith((".jpg", ".png"))]
            hand_counts[hand][class_id] = len(files)
            for fname in files:
                img = cv2.imread(os.path.join(class_dir, fname), cv2.IMREAD_GRAYSCALE)
                if img is None:
                    continue
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                images.append(img)
                labels.append(class_id)
    
    # Fallback: flat layout (dataset/0/, dataset/1/, ...)
    if not images:
        for class_id in range(NUM_CLASSES):
            class_dir = os.path.join(DATASET_DIR, str(class_id))
            if not os.path.exists(class_dir):
                continue
            files = [f for f in os.listdir(class_dir) if f.endswith((".jpg", ".png"))]
            hand_counts["combined"] = hand_counts.get("combined", Counter())
            hand_counts["combined"][class_id] = len(files)
            for fname in files:
                img = cv2.imread(os.path.join(class_dir, fname), cv2.IMREAD_GRAYSCALE)
                if img is None:
                    continue
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                images.append(img)
                labels.append(class_id)
    
    X = np.array(images, dtype=np.float32) / 255.0
    X = X.reshape(-1, IMG_SIZE, IMG_SIZE, 1)
    y = np.array(labels)
    
    return X, y, hand_counts

X, y, hand_counts = load_dataset_with_meta()
print(f"Total samples: {len(X)}")
print(f"Image shape:   {X[0].shape}")
print(f"Classes:        {NUM_CLASSES} (0-5 fingers)")
print(f"\nPer-hand breakdown:")
for hand, counts in hand_counts.items():
    if counts:
        total = sum(counts.values())
        print(f"  {hand.upper()}: {total} images")
        for c in range(NUM_CLASSES):
            print(f"    Class {c}: {counts.get(c, 0)}")

### 1.1 Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall class distribution
class_counts = Counter(y)
classes = list(range(NUM_CLASSES))
counts = [class_counts.get(c, 0) for c in classes]

axes[0].bar(classes, counts, color=NEON, edgecolor=NEON_DIM, alpha=0.85)
axes[0].set_xlabel("Finger Count (Class)")
axes[0].set_ylabel("Number of Images")
axes[0].set_title("Overall Class Distribution", color=NEON)
axes[0].set_xticks(classes)
axes[0].set_xticklabels(CLASS_NAMES, rotation=30, ha="right", fontsize=8)
for i, v in enumerate(counts):
    axes[0].text(i, v + max(counts)*0.02, str(v), ha="center", fontsize=9, color=NEON)

# Per-hand stacked bar chart
left_counts = [hand_counts.get("left", {}).get(c, 0) for c in classes]
right_counts = [hand_counts.get("right", {}).get(c, 0) for c in classes]

if any(left_counts) or any(right_counts):
    x = np.arange(NUM_CLASSES)
    w = 0.35
    axes[1].bar(x - w/2, left_counts, w, label="Left Hand", color="#00AA2A", edgecolor="#005F15")
    axes[1].bar(x + w/2, right_counts, w, label="Right Hand", color=NEON, edgecolor=NEON_DIM)
    axes[1].set_xlabel("Finger Count (Class)")
    axes[1].set_ylabel("Number of Images")
    axes[1].set_title("Per-Hand Distribution", color=NEON)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(CLASS_NAMES, rotation=30, ha="right", fontsize=8)
    axes[1].legend(facecolor="#111", edgecolor="#333", labelcolor="#ccc")
else:
    axes[1].text(0.5, 0.5, "Single-hand dataset\n(no L/R split)", 
                 ha="center", va="center", transform=axes[1].transAxes, fontsize=12)
    axes[1].set_title("Per-Hand Distribution", color=NEON)

plt.tight_layout()
plt.savefig("screenshots/dataset_distribution.png", bbox_inches="tight", facecolor="#0a0a0a")
plt.show()

### 1.2 Sample Images per Class

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(15, 5))

for class_id in range(NUM_CLASSES):
    class_mask = y == class_id
    class_images = X[class_mask]
    
    for row in range(2):
        ax = axes[row, class_id]
        if row < len(class_images):
            idx = np.random.randint(0, len(class_images))
            ax.imshow(class_images[idx].squeeze(), cmap="gray")
        ax.axis("off")
        if row == 0:
            ax.set_title(CLASS_NAMES[class_id], fontsize=9, color=NEON)

fig.suptitle("Sample Images per Gesture Class", fontsize=14, color=NEON, y=1.02)
plt.tight_layout()
plt.savefig("screenshots/sample_images.png", bbox_inches="tight", facecolor="#0a0a0a")
plt.show()

---
## 2. Model Architecture

In [ ]:
model = build_model()
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

total_params = model.count_params()
print(f"\nTotal parameters: {total_params:,}")
print(f"Model input shape: {model.input_shape}")
print(f"Model output shape: {model.output_shape}")

---
## 3. Training

In [ ]:
# Prepare data
y_cat = to_categorical(y, NUM_CLASSES)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_cat, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

# Train
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

callbacks = [
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
    EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True, verbose=1),
]

EPOCHS = 15
BATCH_SIZE = 32

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test, y_test),
    callbacks=callbacks,
)

# Final eval
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nFinal test accuracy: {acc:.1%}")
print(f"Final test loss:     {loss:.4f}")

### 3.1 Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history.history["loss"]) + 1)

# Loss
ax1.plot(epochs_range, history.history["loss"], color=NEON, linewidth=2, label="Train Loss")
ax1.plot(epochs_range, history.history["val_loss"], color="#FF4444", linewidth=2, linestyle="--", label="Val Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training & Validation Loss", color=NEON)
ax1.legend(facecolor="#111", edgecolor="#333", labelcolor="#ccc")
ax1.grid(True, alpha=0.2)
ax1.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# Accuracy
ax2.plot(epochs_range, history.history["accuracy"], color=NEON, linewidth=2, label="Train Acc")
ax2.plot(epochs_range, history.history["val_accuracy"], color="#FF4444", linewidth=2, linestyle="--", label="Val Acc")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Training & Validation Accuracy", color=NEON)
ax2.legend(facecolor="#111", edgecolor="#333", labelcolor="#ccc")
ax2.grid(True, alpha=0.2)
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax2.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig("screenshots/training_curves.png", bbox_inches="tight", facecolor="#0a0a0a")
plt.show()

---
## 4. Classification Metrics

In [ ]:
# Predictions
y_pred_proba = model.predict(X_test)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = np.argmax(y_test, axis=1)

# Classification report
print("Classification Report:")
print("=" * 65)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))

### 4.1 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, interpolation="nearest", cmap="Greens")

ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(CLASS_NAMES, fontsize=9)
ax.set_xlabel("Predicted", fontsize=11)
ax.set_ylabel("True", fontsize=11)
ax.set_title("Confusion Matrix", color=NEON, fontsize=14)

# Annotate cells
thresh = cm.max() / 2.0
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        color = "black" if cm[i, j] > thresh else NEON
        ax.text(j, i, format(cm[i, j], "d"),
                ha="center", va="center", color=color, fontsize=11, fontweight="bold")

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig("screenshots/confusion_matrix.png", bbox_inches="tight", facecolor="#0a0a0a")
plt.show()

### 4.2 Per-Class Precision, Recall & F1-Score

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, average=None)

x = np.arange(NUM_CLASSES)
w = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w, precision, w, label="Precision", color=NEON, alpha=0.9)
ax.bar(x, recall, w, label="Recall", color="#00AA2A", alpha=0.9)
ax.bar(x + w, f1, w, label="F1-Score", color="#005F15", alpha=0.9)

ax.set_xlabel("Gesture Class")
ax.set_ylabel("Score")
ax.set_title("Per-Class Precision, Recall & F1-Score", color=NEON)
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right", fontsize=9)
ax.set_ylim(0, 1.15)
ax.legend(facecolor="#111", edgecolor="#333", labelcolor="#ccc")
ax.grid(True, alpha=0.15, axis="y")

# Annotate F1 values
for i, v in enumerate(f1):
    ax.text(i + w, v + 0.03, f"{v:.2f}", ha="center", fontsize=8, color=NEON)

plt.tight_layout()
plt.savefig("screenshots/precision_recall_f1.png", bbox_inches="tight", facecolor="#0a0a0a")
plt.show()

---
## 5. ROC Curves & AUC (One-vs-Rest)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

colors = ["#00FF41", "#00DD38", "#00BB2F", "#009926", "#00771D", "#005514"]

for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_test[:, i], y_pred_proba[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=colors[i], linewidth=2,
            label=f"{CLASS_NAMES[i]} (AUC = {roc_auc:.3f})")

ax.plot([0, 1], [0, 1], color="#333", linestyle="--", linewidth=1, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — One-vs-Rest", color=NEON, fontsize=14)
ax.legend(loc="lower right", facecolor="#111", edgecolor="#333", labelcolor="#ccc", fontsize=9)
ax.grid(True, alpha=0.15)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])

plt.tight_layout()
plt.savefig("screenshots/roc_curves.png", bbox_inches="tight", facecolor="#0a0a0a")
plt.show()

---
## 6. Prediction Confidence Analysis

In [ ]:
max_confidences = np.max(y_pred_proba, axis=1)
correct_mask = y_pred == y_true

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Confidence distribution for correct vs incorrect predictions
ax1.hist(max_confidences[correct_mask], bins=30, alpha=0.8, color=NEON, label="Correct", edgecolor=NEON_DIM)
ax1.hist(max_confidences[~correct_mask], bins=30, alpha=0.7, color="#FF4444", label="Incorrect", edgecolor="#AA2222")
ax1.set_xlabel("Prediction Confidence")
ax1.set_ylabel("Count")
ax1.set_title("Confidence Distribution: Correct vs Incorrect", color=NEON)
ax1.legend(facecolor="#111", edgecolor="#333", labelcolor="#ccc")
ax1.grid(True, alpha=0.15)

# Per-class average confidence
avg_conf = []
for c in range(NUM_CLASSES):
    mask = y_true == c
    avg_conf.append(max_confidences[mask].mean() if mask.any() else 0)

ax2.barh(range(NUM_CLASSES), avg_conf, color=NEON, edgecolor=NEON_DIM, alpha=0.85)
ax2.set_yticks(range(NUM_CLASSES))
ax2.set_yticklabels(CLASS_NAMES, fontsize=9)
ax2.set_xlabel("Average Confidence")
ax2.set_title("Mean Prediction Confidence per Class", color=NEON)
ax2.set_xlim(0, 1.1)
ax2.grid(True, alpha=0.15, axis="x")
for i, v in enumerate(avg_conf):
    ax2.text(v + 0.02, i, f"{v:.2f}", va="center", fontsize=9, color=NEON)

plt.tight_layout()
plt.savefig("screenshots/confidence_analysis.png", bbox_inches="tight", facecolor="#0a0a0a")
plt.show()

print(f"Overall mean confidence: {max_confidences.mean():.3f}")
print(f"Correct predictions mean confidence: {max_confidences[correct_mask].mean():.3f}")
if (~correct_mask).any():
    print(f"Incorrect predictions mean confidence: {max_confidences[~correct_mask].mean():.3f}")

---
## 7. Inference Latency Benchmark

In [ ]:
import time

dummy_input = np.random.rand(1, IMG_SIZE, IMG_SIZE, 1).astype(np.float32)
N_RUNS = 100

# --- Keras inference ---
# Warm up
model(dummy_input, training=False)

t0 = time.perf_counter()
for _ in range(N_RUNS):
    model(dummy_input, training=False)
keras_time = (time.perf_counter() - t0) / N_RUNS * 1000  # ms

# --- TFLite inference ---
tflite_time = None
if os.path.exists(TFLITE_PATH):
    interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    # Warm up
    interpreter.set_tensor(input_details[0]["index"], dummy_input)
    interpreter.invoke()
    
    t0 = time.perf_counter()
    for _ in range(N_RUNS):
        interpreter.set_tensor(input_details[0]["index"], dummy_input)
        interpreter.invoke()
    tflite_time = (time.perf_counter() - t0) / N_RUNS * 1000  # ms

# --- Results ---
print(f"Inference Latency ({N_RUNS} runs averaged):")
print(f"  Keras direct call:  {keras_time:.2f} ms")
if tflite_time:
    print(f"  TFLite:             {tflite_time:.2f} ms")
    print(f"  Speedup:            {keras_time / tflite_time:.1f}x")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
methods = ["Keras"]
times = [keras_time]
if tflite_time:
    methods.append("TFLite")
    times.append(tflite_time)

bars = ax.barh(methods, times, color=[NEON_DIM, NEON][:len(methods)], edgecolor=NEON, height=0.4)
ax.set_xlabel("Latency (ms)")
ax.set_title("Single-Image Inference Latency", color=NEON)
ax.grid(True, alpha=0.15, axis="x")
for bar, t in zip(bars, times):
    ax.text(t + max(times)*0.03, bar.get_y() + bar.get_height()/2,
            f"{t:.2f} ms", va="center", fontsize=10, color=NEON)

plt.tight_layout()
plt.savefig("screenshots/inference_latency.png", bbox_inches="tight", facecolor="#0a0a0a")
plt.show()

---
## 8. Model Size Comparison

In [ ]:
sizes = {}
if os.path.exists(MODEL_PATH):
    sizes["Keras (.keras)"] = os.path.getsize(MODEL_PATH) / 1024  # KB
if os.path.exists(TFLITE_PATH):
    sizes["TFLite (.tflite)"] = os.path.getsize(TFLITE_PATH) / 1024  # KB

if sizes:
    fig, ax = plt.subplots(figsize=(8, 3))
    names = list(sizes.keys())
    vals = list(sizes.values())
    
    bars = ax.barh(names, vals, color=[NEON_DIM, NEON][:len(names)], edgecolor=NEON, height=0.4)
    ax.set_xlabel("Size (KB)")
    ax.set_title("Model File Size Comparison", color=NEON)
    ax.grid(True, alpha=0.15, axis="x")
    
    for bar, v in zip(bars, vals):
        label = f"{v:.0f} KB" if v < 1024 else f"{v/1024:.1f} MB"
        ax.text(v + max(vals)*0.03, bar.get_y() + bar.get_height()/2,
                label, va="center", fontsize=10, color=NEON)
    
    if len(vals) == 2:
        ratio = vals[0] / vals[1]
        print(f"Keras size:  {vals[0]:.0f} KB")
        print(f"TFLite size: {vals[1]:.0f} KB")
        print(f"Compression: {ratio:.1f}x smaller with TFLite + quantization")
    
    plt.tight_layout()
    plt.savefig("screenshots/model_size.png", bbox_inches="tight", facecolor="#0a0a0a")
    plt.show()
else:
    print("No model files found. Train the model first.")

---
## 9. Waveform Visualizations

Visual comparison of the three waveform types generated by the synthesizer at 440 Hz (A4).

In [ ]:
freq = 440.0  # A4
duration = 0.005  # 5ms — ~2 full cycles at 440Hz
t = np.linspace(0, duration, int(SAMPLE_RATE * duration))

waveforms = {
    "Sine": generate_sine(t, freq),
    "Square": generate_square(t, freq),
    "Sawtooth": generate_sawtooth(t, freq),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, samples) in zip(axes, waveforms.items()):
    ax.plot(t * 1000, samples, color=NEON, linewidth=1.5)
    ax.fill_between(t * 1000, samples, alpha=0.15, color=NEON)
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Amplitude")
    ax.set_title(f"{name} Wave — {freq:.0f} Hz", color=NEON)
    ax.set_ylim(-1.3, 1.3)
    ax.grid(True, alpha=0.15)
    ax.axhline(y=0, color="#333", linewidth=0.5)

plt.tight_layout()
plt.savefig("screenshots/waveforms.png", bbox_inches="tight", facecolor="#0a0a0a")
plt.show()

---
## 10. Frequency Spectrum (FFT) of Each Waveform

Shows the harmonic content of each waveform type — important for understanding timbral differences.

In [ ]:
freq = 440.0
duration = 0.1  # longer sample for better frequency resolution
t = np.linspace(0, duration, int(SAMPLE_RATE * duration))

waveforms_fft = {
    "Sine": generate_sine(t, freq),
    "Square": generate_square(t, freq),
    "Sawtooth": generate_sawtooth(t, freq),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, samples) in zip(axes, waveforms_fft.items()):
    n = len(samples)
    fft_vals = np.abs(np.fft.rfft(samples)) / n
    fft_freqs = np.fft.rfftfreq(n, 1.0 / SAMPLE_RATE)
    
    # Only show up to 5kHz
    mask = fft_freqs <= 5000
    ax.plot(fft_freqs[mask], fft_vals[mask], color=NEON, linewidth=1)
    ax.fill_between(fft_freqs[mask], fft_vals[mask], alpha=0.2, color=NEON)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Magnitude")
    ax.set_title(f"{name} — Frequency Spectrum", color=NEON)
    ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.savefig("screenshots/fft_spectrum.png", bbox_inches="tight", facecolor="#0a0a0a")
plt.show()

---
## 11. Misclassified Samples

Visualize samples the model got wrong — useful for error analysis in a research paper.

In [ ]:
misclassified_idx = np.where(y_pred != y_true)[0]
n_show = min(12, len(misclassified_idx))

if n_show > 0:
    fig, axes = plt.subplots(2, min(6, n_show), figsize=(15, 5))
    if n_show <= 6:
        axes = axes.reshape(1, -1) if n_show > 1 else np.array([[axes]])
    
    # Randomly sample misclassified images
    show_idx = np.random.choice(misclassified_idx, n_show, replace=False)
    
    for i, idx in enumerate(show_idx):
        row, col = divmod(i, 6)
        if n_show <= 6:
            ax = axes[0, i]
        else:
            ax = axes[row, col]
        ax.imshow(X_test[idx].squeeze(), cmap="gray")
        conf = max_confidences[idx]
        ax.set_title(f"True: {y_true[idx]}\nPred: {y_pred[idx]} ({conf:.0%})",
                     fontsize=8, color="#FF4444")
        ax.axis("off")
    
    # Hide empty subplots
    for i in range(n_show, axes.size):
        row, col = divmod(i, 6)
        if n_show <= 6:
            axes[0, i].axis("off") if i < axes.shape[1] else None
        else:
            axes[row, col].axis("off")
    
    fig.suptitle(f"Misclassified Samples ({len(misclassified_idx)} total errors out of {len(y_true)} test samples)",
                 fontsize=12, color="#FF4444", y=1.02)
    plt.tight_layout()
    plt.savefig("screenshots/misclassified.png", bbox_inches="tight", facecolor="#0a0a0a")
    plt.show()
else:
    print("No misclassifications — perfect accuracy on test set!")

---
## 12. Summary Table

Key metrics at a glance for the paper's results section.

In [ ]:
from sklearn.metrics import accuracy_score

macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro")
weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted")

keras_size = os.path.getsize(MODEL_PATH) / 1024 if os.path.exists(MODEL_PATH) else 0
tflite_size = os.path.getsize(TFLITE_PATH) / 1024 if os.path.exists(TFLITE_PATH) else 0

print("=" * 60)
print("  SUMMARY — Key Metrics for Research Paper")
print("=" * 60)
print()
print(f"  Dataset")
print(f"    Total samples:         {len(X)}")
print(f"    Training set:          {len(X_train)}")
print(f"    Test set:              {len(X_test)}")
print(f"    Number of classes:     {NUM_CLASSES}")
print(f"    Input resolution:      {IMG_SIZE}x{IMG_SIZE} grayscale")
print()
print(f"  Model")
print(f"    Architecture:          3-layer CNN")
print(f"    Total parameters:      {model.count_params():,}")
print(f"    Keras model size:      {keras_size:.0f} KB")
print(f"    TFLite model size:     {tflite_size:.0f} KB")
print()
print(f"  Training")
print(f"    Epochs (max):          {EPOCHS}")
print(f"    Epochs (actual):       {len(history.history['loss'])}")
print(f"    Batch size:            {BATCH_SIZE}")
print(f"    Optimizer:             Adam")
print(f"    Loss function:         Categorical Cross-Entropy")
print()
print(f"  Classification Performance")
print(f"    Test accuracy:         {acc:.1%}")
print(f"    Test loss:             {loss:.4f}")
print(f"    Macro precision:       {macro_p:.3f}")
print(f"    Macro recall:          {macro_r:.3f}")
print(f"    Macro F1-score:        {macro_f1:.3f}")
print(f"    Weighted F1-score:     {weighted_f1:.3f}")
print()
print(f"  Inference Latency")
print(f"    Keras:                 {keras_time:.2f} ms")
if tflite_time:
    print(f"    TFLite:                {tflite_time:.2f} ms")
    print(f"    Speedup:               {keras_time/tflite_time:.1f}x")
print()
print("=" * 60)